In [5]:
# ============================================================================
# STEP 0: SSL CERT AND ENVIRONMENT
# ============================================================================
import certifi
import os
from dotenv import load_dotenv

# Set SSL certificate for HTTPS connections
os.environ["SSL_CERT_FILE"] = certifi.where()
print("SSL_CERT_FILE set to:", os.environ["SSL_CERT_FILE"])

# Load environment variables from .env
load_dotenv()


SSL_CERT_FILE set to: D:\Data_Science\CV\Lib\site-packages\certifi\cacert.pem


True

In [6]:
# Importing
import os #For env
import re
from langchain_groq import ChatGroq
from langchain_community.utilities import SQLDatabase
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
import datetime

In [7]:
# Init the model
llm_for_reasoning = ChatGroq(api_key = os.getenv("GROQ_API_KEY"),
               model_name = "llama-3.3-70b-versatile",
               temperature = 0)

In [8]:
import sqlite3
# Run this first, init the database
def init_sqlite_database():
    """init from init_sqlite.sql"""
    try:
        db_path = "attendance.db"

        if not os.path.exists("init_sqlitedb.sql"):
            print("No init file")
            return None

        #init because found

        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()

        # Reading and executing the file
        with open("init_sqlitedb.sql","r") as f:
            sql_script = f.read()

        cursor.executescript(sql_script)
        conn.commit() #Saving

        cursor.execute("SELECT name FROM sqlite_master WHERE type = 'table';")
        tables = [table[0] for table in cursor.fetchall()]

        conn.close()
        print(f"Create SQLite db with tables: {tables}")
        return SQLDatabase.from_uri(f"sqlite:///{db_path}")
    except Exception as e:
        print("Failed to init sql db:")
        print(e)
        return None
def test_connection():
    try:
        db = SQLDatabase.from_uri("sqlite:///attendance.db")
        tables = db.get_usable_table_names()
        print(f"Connected, found: {tables}")
        
        result = db.run("SELECT COUNT(*) as student_count FROM students")
        print(f"📊 Students in database: {result}")
        return db
    except Exception as e:
        print(f"Connection failed: {e}")
        print("Creating a new sql db")
        return init_sqlite_database()

In [9]:
db = test_connection()

Connected, found: []
Connection failed: (sqlite3.OperationalError) no such table: students
[SQL: SELECT COUNT(*) as student_count FROM students]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
Creating a new sql db
Create SQLite db with tables: ['students', 'sqlite_sequence', 'attendance_sessions', 'daily_attendance', 'class_schedule', 'semester_config', 'student_circumstances']


## Helper functions

In [10]:
def parse_ai_response(ai_response, expected_values):
    """
    Parse AI response looking for lines with exactly (expected_values - 1) commas
    and exactly expected_values parts when split by commas.
    
    Args:
        ai_response (str): The raw AI response
        expected_values (int): Number of expected values (e.g., 3 for status,score,reason)
    
    Returns:
        list: Parsed values if found, None if no valid line found
    """
    try:
        expected_commas = expected_values - 1
        
        lines = ai_response.split('\n')
        for line in lines:
            line = line.strip()
            
            # Skip empty lines
            if not line:
                continue
                
            # Count commas in this line
            comma_count = line.count(',')
            
            if comma_count == expected_commas:
                parts = line.split(',')
                
                # Check if we have exactly the right number of parts
                if len(parts) == expected_values:
                    # All parts should have content (not empty after stripping)
                    cleaned_parts = [part.strip() for part in parts]
                    if all(cleaned_parts):
                        return cleaned_parts
        
        return None
        
    except Exception as e:
        logging.error(f"Error parsing AI response: {e}")
        return None

In [11]:
from datetime import datetime, time
import logging

def get_connection(row_factory = None):
    """Connect to db"""
    conn = sqlite3.connect("attendance.db")
    if row_factory:
        conn.row_factory = row_factory
    return conn
def get_current_session_direct():
    """
    Get the current session, next one if it's break time
    """
    conn = get_connection()
    try:
        cursor = conn.cursor()
        query = """
            SELECT session_number, start_time, end_time
            FROM class_schedule
            WHERE start_time<=TIME('now','localtime') AND TIME('now','localtime') <=end_time
            LIMIT 1
        """
        cursor.execute(query)
        result = cursor.fetchone()
        if result is None:
            query = """
                SELECT session_number, start_time, end_time
                FROM class_schedule
                WHERE TIME('now','localtime') < start_time
                ORDER BY session_number ASC
                LIMIT 1
            """
            cursor.execute(query)
            result = cursor.fetchone()
        if result:
            # Convert string times to time
            session_number, start_str, end_str = result
            start_time = datetime.strptime(start_str, '%H:%M:%S').time()
            end_time = datetime.strptime(end_str, '%H:%M:%S').time()

            return {
                'session_number':session_number,
                'start_time':start_time,
                'end_time':end_time
            }
        return "No active session"
    finally:
        conn.close()
def get_current_session_with_time(input_time):
    """
    Get the current session based on a specific input time
    """
    conn = get_connection()
    try:
        cursor = conn.cursor()
        
        time_str = input_time.strftime('%H:%M:%S')
        
        query = """
            SELECT session_number, start_time, end_time
            FROM class_schedule
            WHERE start_time <= ? AND ? <= end_time
            LIMIT 1
        """
        cursor.execute(query, (time_str, time_str))
        result = cursor.fetchone()
        
        if result is None:
            query = """
                SELECT session_number, start_time, end_time
                FROM class_schedule
                WHERE ? < start_time
                ORDER BY session_number ASC
                LIMIT 1
            """
            cursor.execute(query, (time_str,))
            result = cursor.fetchone()
            
        if result:
            session_number, start_str, end_str = result
            start_time = datetime.strptime(start_str, '%H:%M:%S').time()
            end_time = datetime.strptime(end_str, '%H:%M:%S').time()

            return {
                'session_number': session_number,
                'start_time': start_time,
                'end_time': end_time
            }
        return "No active session"
    finally:
        conn.close()

In [12]:
def get_session_by_number(session_number:int):
    """Get the session by nunmber"""
    conn = get_connection()
    try:
        cursor = conn.cursor()
        cursor.execute("""
        SELECT session_number, start_time, end_time
        FROM class_schedule
        WHERE session_number = ?
        """,(session_number,))

        result = cursor.fetchone()
        if result:
            session_num, start_str, end_str = result
            start_time = datetime.strptime(start_str, '%H:%M:%S').time()
            end_time = datetime.strptime(end_str, '%H:%M:%S').time()
            return {
                'session_number':session_num,
                'start_time':start_time,
                'end_time':end_time
            }
        return None
    finally:
        conn.close()

In [13]:
def calculate_late_minutes (entry_time, scheduled_start_time):
    """ calculate late minutes"""
    if isinstance(entry_time, datetime):
        entry_time = entry_time.time()
    if isinstance(scheduled_start_time, str):
        scheduled_start_time = datetime.strptime(scheduled_start_time, '%H:%M:%S').time()

    entry_datetime = datetime.combine(datetime.today(), entry_time)
    scheduled_datetime = datetime.combine(datetime.today(), scheduled_start_time)

    if entry_datetime > scheduled_datetime:
        delta = entry_datetime - scheduled_datetime
        return int(delta.total_seconds()/60)
    return 0


In [14]:
def get_or_create_student(name):
    """Get or create student """
    try:
        conn = get_connection()
        cursor = conn.cursor()
        
        cursor.execute("SELECT id FROM students WHERE name = ?", (name,))
        result = cursor.fetchone()

        if not result:
            cursor.execute("INSERT INTO students (name) VALUES (?)", (name,))
            student_id = cursor.lastrowid
            conn.commit()
            logging.info(f"Created new student: {name} (ID: {student_id})")
        else:
            student_id = result[0]
            logging.info(f"Found existing student: {name} (ID: {student_id})")

        conn.close()
        return student_id
        
    except Exception as e:
        logging.error(f"Error getting/creating student: {e}")
        return None


In [15]:
def get_student_attendance_history(student_id):
    """Get the student history"""
    try:
        with get_connection() as conn:
            cursor = conn.cursor()
            cursor.execute("""
                SELECT 
                    COUNT(*) as total_sessions,
                    SUM(CASE WHEN attendance_status = 'late' THEN 1 ELSE 0 END) as late_count,
                    SUM(CASE WHEN attendance_status = 'very_late' THEN 1 ELSE 0 END) as very_late_count,
                    AVG(late_minutes) as avg_late_minutes
                FROM attendance_sessions 
                WHERE student_id = ? 
                AND session_date >= DATE('now', '-7 days')
            """, (student_id,))
            result = cursor.fetchone()
            if result:
                total,late,very_late, avg_late = result
                # Cho cac phan tu bang None trong SQLITE
                late = late or 0
                very_late = very_late or 0
                avg_late = avg_late or 0

                return f"Last 7 days: {total} sessions, {late} late, {very_late} very late, avg{avg_late:.1f} min late"
            return "No recent history"
    except Exception as e:
        logging.error(f"Error while getting student hisotry:")
        return "History unavailable"

def active_check(student_id):
    """If the date is valid, active, if not, deactive"""
    try:
        
        with get_connection() as conn:
            #deactive
            cursor = conn.cursor()
            cursor.execute("""
                SELECT id, start_date, end_date
                FROM student_circumstances 
                WHERE student_id = ? AND is_active = 1
            """, (student_id,))
            results_active = cursor.fetchall()
            cursor.execute("""
                SELECT id, start_date, end_date
                FROM student_circumstances 
                WHERE student_id = ? AND is_active = 0
            """, (student_id,))
            results_deactive = cursor.fetchall()
            for result in results_active:
                cir_id , start_date, end_date = result
                start_date = datetime.strptime(start_date, "%Y-%m-%d")
                end_date = datetime.strptime(end_date, "%Y-%m-%d")
                current_date = datetime.now()
                if current_date>end_date or current_date<start_date:
                    cursor.execute("""
                        UPDATE student_circumstances
                        SET is_active = 0
                        WHERE id = ?
                    """,(cir_id,))
            #active
            for result in results_deactive:
                cir_id , start_date, end_date = result
                start_date = datetime.strptime(start_date, "%Y-%m-%d")
                end_date = datetime.strptime(end_date, "%Y-%m-%d")
                current_date = datetime.now()
                if current_date<=end_date and current_date>=start_date:
                    cursor.execute("""
                        UPDATE student_circumstances
                        SET is_active = 1
                        WHERE id = ?
                    """,(cir_id,))       
    except Exception as e:
        print(e)



def get_student_circumstances(student_id, session_number=None):
    """Get student circumstances with session-specific excuses"""
    try:
        with get_connection() as conn:
            cursor = conn.cursor()
            active_check(student_id)
            if session_number:
                # Get circumstances specific to this session
                cursor.execute("""
                    SELECT circumstance_type, description, session_numbers, excuse_type
                    FROM student_circumstances 
                    WHERE student_id = ? AND is_active = 1
                    AND date('now') BETWEEN start_date AND end_date
                    AND (session_numbers = 'all' OR session_numbers LIKE ?)
                """, (student_id, f'%{session_number}%'))
            else:
                # Get all active circumstances
                cursor.execute("""
                    SELECT circumstance_type, description, session_numbers, excuse_type
                    FROM student_circumstances 
                    WHERE student_id = ? AND is_active = 1
                    AND date('now') BETWEEN start_date AND end_date
                """, (student_id,))

            results = cursor.fetchall()
            if results:
                circumstances = []
                for circ_type, description, session_nums, excuse_type in results:
                    if session_nums and excuse_type:
                        circumstances.append(f"{circ_type}({excuse_type} for sessions:{session_nums}):{description}")
                    else:
                        circumstances.append(f"{circ_type}:{description}")
                return " | ".join(circumstances)
            return "No active circumstances"
    except Exception as e:
        logging.error(f"Error getting student circumstances: {e}")
        return "Circumstances unavailable"

In [16]:
#Test
active_check(1)

In [17]:
#Test 
get_student_attendance_history(1)
get_student_circumstances(1,1)

'No active circumstances'

In [24]:
def calculate_auto_fill_score(student_id, session_num, llm, student_name, is_entry):
    """Determining the score based on circumstances and stuff"""

    student_circumstances = get_student_circumstances(student_id, session_num)
    if is_entry:
        # Check circumstances for sessions BEFORE current
        ai_prompt = f"""
        AUTO-FILL SCORING FOR ENTRY:
        
        STUDENT: {student_name}
        SESSION: {session_num} (session being auto-filled)
        FIRST ENTRY: TRUE
        CIRCUMSTANCES: {student_circumstances}
        
        SCORING RULES FOR SESSIONS BEFORE ENTRY (STRICT - FOLLOW EXACTLY):
        - 'excused' (score: 1.0): Student has 'full' excuse for this specific session
        - 'excused' (score: 0.5): Student has 'partial' or 'late_arrival' excuse for this session  
        - 'absent' (score: 0.0): No valid documented excuse for this session
        
        STATUS RULES FOR FIRST ENTRY:
        • Use 'excused' ONLY if circumstances specifically mention this session number
        • Use 'absent' if no valid excuse exists
        • DO NOT use 'on_time' or 'late' for auto-filled first entry sessions
        
        Return ONLY: status,score,reason
        Valid Examples:
        excused,1.0,has_medical_excuse_for_session_{session_num}
        excused,0.5,has_transportation_issues_documented
        absent,0.0,no_documented_excuse_for_this_session
        
        Your decision (ONLY use 'excused' or 'absent'):
        """
    else:
        # Between last entry and current exit
        ai_prompt = f"""
        AUTO-FILL SCORING FOR MISSED SESSION:
        
        STUDENT: {student_name}
        SESSION: {session_num} (session being auto-filled)
        CIRCUMSTANCES: {student_circumstances}
        
        SCORING RULES FOR MISSED SESSIONS BETWEEN RECORDED ENTRIES (STRICT - FOLLOW EXACTLY):
        - 'on_time' (score: 1.0): Assume student was present but forgot to record entry (default)
        - 'absent' (score: 0.0): Only if clear evidence of absence from circumstances
        
        STATUS RULES FOR MISSED SESSIONS:
        • Use 'on_time' as default assumption (student was present between scans)
        • Use 'absent' ONLY if clear evidence they were missing
        • DO NOT use 'late' or 'excused' for auto-filled between sessions
        
        Return ONLY: status,score,reason
        Valid Examples:
        on_time,1.0,assumed_present_between_recorded_sessions
        on_time,1.0,student_likely_present_based_on_movement_pattern
        absent,0.0,clear_evidence_of_absence_from_circumstances
        
        Your decision (ONLY use 'on_time' or 'absent'):
        """
    
    ai_response = llm.invoke(ai_prompt).content.strip()
    print(f"AI Response: {ai_response}")
    
    parsed_values = parse_ai_response(ai_response, 3)
    
    if parsed_values and len(parsed_values) == 3:
        status, score_str, reason = parsed_values
        try:
            score = float(score_str)
            
            if is_entry:
                allowed_statuses = ['excused', 'absent']
            else:
                allowed_statuses  = ['on_time', 'absent']
            
            if status not in allowed_statuses:
                logging.warning(f"Invalid status '{status}' for context, using fallback")
                status = 'absent' if is_entry else 'on_time'
                score = 0.0 if is_entry else 1.0
                reason = f'fallback_invalid_status_{reason}'
            
            # Validate score range
            if score < 0 or score > 1:
                score = max(0.0, min(1.0, score))
                
            return {
                'status': status,
                'score': score,
                'reason': reason
            }
            
        except ValueError:
            logging.warning(f"Invalid score format: {score_str}")
    
    # Fallback if parsing fails - ALWAYS return a dictionary
    logging.warning(f"Could not parse AI auto-fill response, using default")
    fallback_status = 'absent' if is_entry else 'on_time'
    fallback_score = 0.0 if is_entry else 1.0
    
    return {
        'status': fallback_status,
        'score': fallback_score,
        'reason': 'auto_fill_parse_error_using_default'
    }

In [25]:
def auto_fill_missing_sessions(student_id,last_session_num, current_session_num, llm, student_name, is_entry):
    """ Auto fill the session before the first entry
    and the session between the last entry and the nearest exist"""
    try:
        with get_connection() as conn:
            cursor = conn.cursor()
            filled_sessions = []
            current_date = datetime.now().strftime('%Y-%m-%d')
            # #Get the last session_num from database
            # cursor.execute("""
            #     SELECT MAX(CAST(session_number AS INTEGER)) 
            #     FROM attendance_sessions 
            #     WHERE student_id = ? AND session_date = ?
            # """, (student_id, current_date))
            # result = cursor.fetchone()
            #If none session, default to 0
            # last_session_num = result[0] if result[0] else 0
            print(f"Last session num was: {last_session_num}")
            print(f"current session num was: {current_session_num}")
            
            for session_num in range(last_session_num + 1, current_session_num):
                session_info = get_session_by_number(session_num)
                print(f"Checking session {session_num}, filled_sessions: {filled_sessions}")
                if session_info:
                    auto_fill_result = calculate_auto_fill_score(student_id, session_num, llm, student_name, is_entry)
                    
                    # Use the dictionary values correctly
                    status = auto_fill_result['status']
                    score = auto_fill_result['score']
                    reason = auto_fill_result['reason']
                    

                    cursor.execute("""
                        INSERT INTO attendance_sessions 
                        (student_id, session_date, entry_time, status, attendance_status, 
                         session_number, reason_for_scoring,attendance_score, late_minutes)
                        VALUES (?, ?, ?, ?, ?, ?, 
                               ?,?, 0)
                    """, (
                        student_id, current_date, 
                        datetime.now().strftime('%H:%M:%S'),
                        "present" if status!="absent" and is_entry==True else "left",
                        status,
                        session_num,
                        f"AUTO_FILLED: {reason} (score:{score})",
                        score
                    ))                    
                    filled_sessions.append({
                        'session': session_num,
                        'status': status,
                        'score': score,
                        'reason': reason
                    })

                    logging.info(f"Auto-filled session {session_num} for {student_name}, ID: {student_id} with status '{status}' and score {score}")
            
            conn.commit()
            return filled_sessions

    except Exception as e:
        logging.error(f"Auto fill score failed: {e}")
        return []

In [28]:
def record_entry(name, llm , active_session = None):
    """Recording Entry. Note: the none values are just for debugging"""
    try:
        student_id = get_or_create_student(name)
        if student_id is None:
            return None
        with get_connection() as conn:
            cursor = conn.cursor()
            current_datetime = datetime.now()
            current_date = current_datetime.strftime('%Y-%m-%d')
            current_time = current_datetime.time()
            session_info = get_current_session_direct()
            # Find the last exit_time
            cursor.execute("""
                SELECT exit_time
                FROM attendance_sessions
                WHERE student_id = ? AND session_date = ? AND exit_time IS NOT NULL
                ORDER BY exit_time DESC
                LIMIT 1
            """, (student_id, current_date))
            
            result = cursor.fetchone()
            if result and result[0] is not None:
                # Convert string to time object
                last_exit_time_str = result[0]
                last_exit_time = datetime.strptime(last_exit_time_str, '%H:%M:%S').time()
            else:
                last_exit_time = None

            
            if active_session is not None:
                if isinstance(active_session, int):
                    session_info = get_session_by_number(active_session)
                else:
                    session_info = active_session
                print(f"After override: Session {session_info}") 
                
            if not session_info:
                logging.warning("No active session found")
                return None

            # Get the session number from the last exit time
            if last_exit_time is not None:
                last_session_info = get_current_session_with_time(last_exit_time)
                last_session_num = last_session_info['session_number'] if isinstance(last_session_info, dict) else 0
            else:
                last_session_num = 0
                    
            #If last lession_num is zero -> first_entry
    
            current_session_num = session_info['session_number']

            #Not allowing a second entry
            if last_session_num ==current_session_num:
                return None
            
            # Auto filling for entry
            auto_fill_missing_sessions(student_id, last_session_num, current_session_num, llm,name, True)
        

            #To rework from here!!
            
            late_minutes = calculate_late_minutes(current_time, session_info['start_time'])

            # Get circumstances for THIS specific session
            student_history = get_student_attendance_history(student_id)
            student_circumstances = get_student_circumstances(student_id, session_info['session_number'])
            
            ai_prompt = f"""
            ATTENDANCE DECISION MAKING:
            
            STUDENT PROFILE:
            - Name: {name} (ID: {student_id})
            - Current Time: {current_time}
            - Session: {session_info['session_number']} ({session_info['start_time']}-{session_info['end_time']})
            - Late by: {late_minutes} minutes
            
            HISTORICAL CONTEXT:
            {student_history}
            
            PERSONAL CIRCUMSTANCES:
            {student_circumstances}
            
            DECISION MATRIX (STRICT RULES - FOLLOW EXACTLY):
            - 'on_time' (score: 1.0): Arrived within 5 minutes of session start
            - 'late' (score: 0.1-0.9): Arrived 5-60 minutes late, adjust score based on circumstances
            - 'absent' (score: 0.0): Arrived 60+ minutes late OR no valid circumstances for extreme lateness
            - 'excused' (score: 1.0): Has valid documented excuse for this specific session
            
            EXCUSE RULES:
            • Use 'excused' ONLY if circumstances specifically mention this session number
            • 'full' excuse type = completely excused regardless of arrival time
            • 'late_arrival' excuse type = excused for being late to this session
            
            SCORING GUIDELINES FOR 'late' STATUS:
            • 0.8-0.9: 5-15 min late with valid circumstances
            • 0.6-0.7: 15-30 min late with mitigating factors  
            • 0.4-0.5: 30-45 min late with minor circumstances
            • 0.1-0.3: 45-60 min late with weak or no valid reasons
            
            Return ONLY: status,score,reason_for_scoring
            Valid Examples:
            on_time,1.0,arrived_within_5_minute_grace_period
            late,0.8,15_min_late_due_to_documented_medical_appointment
            excused,1.0,has_medical_excuse_for_session_3
            absent,0.0,75_min_late_no_valid_circumstances
            
            Your decision (ONLY use 'on_time', 'late', 'absent', or 'excused'):
            """
            ai_response = llm.invoke(ai_prompt).content.strip()
            print(ai_response)
            # Parsing and checking for error
            parsed_values = parse_ai_response(ai_response, 3)
            if not parsed_values:
                logging.warning(f"Failed to parse AI response: {ai_response}, using fallback")
                if late_minutes <= 5:
                    status, score, reason_for_scoring = 'on_time', 1.0, 'fallback_grace_period'
                elif late_minutes <= 60:
                    status, score, reason_for_scoring = 'late', max(0.1, 1.0 - (late_minutes / 60)), 'fallback_late'
                else:
                    status, score, reason_for_scoring = 'absent', 0.0, 'fallback_absent'
            else:
                status, score, reason_for_scoring = parsed_values
            #Constraints
            allowed_statuses = ['on_time', 'late', 'absent', 'excused']
            if status not in allowed_statuses:
                logging.warning(f"AI returned invalid status: {status}, defaulting to 'late'")
                status = 'late'
            
            
            attendance_status = status
            attendance_score = float(score)
            ai_reason = reason_for_scoring

            if late_minutes > 0:
                logging.warning(f"🚨 {name} is LATE by {late_minutes} minutes!")
                print(f"\n{'=' * 80}")
                print(f"🤖 ADVANCED AI ATTENDANCE DECISION ENGINE")
                print(f"{'=' * 80}")
                print(f"👤 Student: {name} (ID: {student_id})")
                print(f"⏰ Scheduled: {session_info['start_time']}")
                print(f"🕒 Arrived: {current_time.strftime('%H:%M:%S')}")
                print(f"⚠️  Late by: {late_minutes} minutes")
                print(f"📊 Historical Pattern: {student_history}")
                print(f"🎯 Personal Circumstances: {student_circumstances}")
                print(f"🤖 AI Decision: {attendance_status}")
                print(f"⭐ AI Score: {attendance_score}")
                print(f"💡 Reason for Scoring: {ai_reason}")
                print(f"{'=' * 80}\n")
            else:
                logging.info(f"✅ {name} - AI Decision: {attendance_status}")
                print(f"\n{'=' * 60}")
                print(f"✅ AI ATTENDANCE CONFIRMED")
                print(f"{'=' * 60}")
                print(f"Student: {name}")
                print(f"Status: {attendance_status}")
                print(f"Score: {attendance_score}")
                print(f"Reason: {ai_reason}")
                print(f"{'=' * 60}\n")

            # Store only time in entry_time, not full datetime
            cursor.execute("""
                INSERT INTO attendance_sessions 
                (student_id, session_date, entry_time, status, attendance_status, late_minutes, reason_for_scoring,attendance_score, session_number)
                VALUES (?, ?, ?, 'present', ?, ?, ?,?, ?)
            """, (
                student_id, 
                current_date,  # Date goes here
                current_time.strftime('%H:%M:%S'), 
                attendance_status, 
                late_minutes, 
                ai_reason,
                attendance_score,
                session_info['session_number']  # ADD session_number
            ))

            session_id = cursor.lastrowid
            conn.commit()

            logging.info(f"🤖 Advanced AI attendance recorded for {name}: {attendance_status} - {ai_reason}")
            
            return {
                'session_id': session_id,
                'student_id': student_id,
                'status': attendance_status,
                'score': attendance_score,
                'late_minutes': late_minutes,
                'reason_for_scoring': ai_reason,
                'timestamp': current_datetime,
                'circumstances_considered': student_circumstances,
                'historical_context': student_history
            }
    except Exception as e:
        print(e)
        return None
                

In [29]:
record_entry("Emily Johnson", llm_for_reasoning, 4)

After override: Session {'session_number': 4, 'start_time': datetime.time(9, 55), 'end_time': datetime.time(10, 40)}
Last session num was: 0
current session num was: 4
Checking session 1, filled_sessions: []
AI Response: absent,0.0,no_documented_excuse_for_this_session
Checking session 2, filled_sessions: [{'session': 1, 'status': 'absent', 'score': 0.0, 'reason': 'no_documented_excuse_for_this_session'}]
AI Response: absent,0.0,no_documented_excuse_for_this_session
Checking session 3, filled_sessions: [{'session': 1, 'status': 'absent', 'score': 0.0, 'reason': 'no_documented_excuse_for_this_session'}, {'session': 2, 'status': 'absent', 'score': 0.0, 'reason': 'no_documented_excuse_for_this_session'}]
AI Response: absent,0.0,no_documented_excuse_for_this_session


absent,0.0,461_min_late_no_valid_circumstances

🤖 ADVANCED AI ATTENDANCE DECISION ENGINE
👤 Student: Emily Johnson (ID: 2)
⏰ Scheduled: 09:55:00
🕒 Arrived: 17:36:14
⚠️  Late by: 461 minutes
📊 Historical Pattern: Last 7 days: 13 sessions, 0 late, 0 very late, avg35.2 min late
🎯 Personal Circumstances: No active circumstances
🤖 AI Decision: absent
⭐ AI Score: 0.0
💡 Reason for Scoring: 461_min_late_no_valid_circumstances



{'session_id': 94,
 'student_id': 2,
 'status': 'absent',
 'score': 0.0,
 'late_minutes': 461,
 'reason_for_scoring': '461_min_late_no_valid_circumstances',
 'timestamp': datetime.datetime(2025, 11, 26, 17, 36, 14, 818204),
 'circumstances_considered': 'No active circumstances',
 'historical_context': 'Last 7 days: 13 sessions, 0 late, 0 very late, avg35.2 min late'}

In [30]:
def record_exit(name, llm, active_session=None, early_departure_reason=None):
    """Record exit with auto-filling. Note: the none values are just for debugging"""
    try:
        student_id = get_or_create_student(name)
        if student_id is None:
            return None
        with get_connection() as conn:
            cursor = conn.cursor()
            current_date = datetime.now().strftime('%Y-%m-%d')
            current_time = datetime.now().time()
            session_info = get_current_session_direct()

            # Find the last entry time for auto filling
            cursor.execute("""
                SELECT entry_time, session_number
                FROM attendance_sessions
                WHERE student_id = ? AND session_date = ?
                ORDER BY session_number DESC, entry_time DESC
                LIMIT 1
            """, (student_id, current_date))

            result = cursor.fetchone()

            if result and result[0] is not None:
                # Convert string to time object
                last_entry_time_str = result[0]
                current_session_id = result[1]  # Get the session ID for the UPDATE
                last_entry_time = datetime.strptime(last_entry_time_str, '%H:%M:%S').time()
            else:
                last_entry_time = None
                current_session_id = None
            print(last_entry_time)
            if active_session is not None:
                if isinstance(active_session, int):
                    session_info = get_session_by_number(active_session)
                else:
                    session_info = active_session

            if not session_info:
                logging.warning("The class has already ended!!")
                return None

            # Get the session number from the last entry time
            if last_entry_time is not None:
                last_session_info = get_current_session_with_time(last_entry_time)
                last_session_num = last_session_info['session_number'] if isinstance(last_session_info, dict) else 0
            else:
                last_session_num = 0

            current_session_num = session_info["session_number"]

            # Auto filling for exit
            auto_fill_missing_sessions(student_id, last_session_num, current_session_num, llm, name, False) # Not an entry
            
            # Calculating score part
            entry_time = last_entry_time  
            current_datetime = datetime.now() 
            entry_datetime = datetime.combine(current_datetime.date(), entry_time)     
            
            # Early leaving duration
            duration_minutes = max(1, int((current_datetime - entry_datetime).total_seconds() / 60))
            session_end_time = session_info['end_time']  
            early_departure_minutes = 0
            
            if current_time < session_end_time:
                end_dt = datetime.combine(current_datetime.date(), session_end_time)
                early_departure_minutes = max(0, int((end_dt - current_datetime).total_seconds() / 60))
            
            student_history = get_student_attendance_history(student_id)
            student_circumstances = get_student_circumstances(student_id, session_info['session_number'])
            
            ai_prompt = f"""
            EARLY DEPARTURE PENALTY CALCULATION:
            
            STUDENT: {name}
            SESSION: {session_info['session_number']} ({session_info['start_time']}-{session_info['end_time']})
            EARLY DEPARTURE: {early_departure_minutes} minutes early
            REASON: {early_departure_reason or 'Not specified'}
            
            HISTORICAL PATTERNS:
            {student_history}
            
            CIRCUMSTANCES:
            {student_circumstances}
            
            PENALTY MATRIX:
            - 0-5 min early: 10% penalty (score: 0.9)
            - 6-15 min early: 30% penalty (score: 0.7)  
            - 16-30 min early: 60% penalty (score: 0.4)
            - 31+ min early: 90% penalty (score: 0.1)
            - Medical/emergency: No penalty (score: 1.0)
            - Pre-approved: Reduced penalty
            
            IMPORTANT: Respond ONLY in this exact format: final_score,penalty_reason
            - final_score: number between 0.1 and 1.0
            - penalty_reason: short_description_without_spaces
            
            Examples:
            0.7,left_12_minutes_early_30_percent_penalty
            1.0,medical_appointment_no_penalty
            0.4,left_25_minutes_early_60_percent_penalty
            0.9,left_3_minutes_early_10_percent_penalty
            
            Do NOT include any explanations, just the score and reason separated by comma.
            
            Your response:
            """
            
            ai_response = llm.invoke(ai_prompt).content.strip()

            # Parsing
            final_score, penalty_reason = parse_ai_response(ai_response, 2)
            try:
                final_score_float = float(final_score)
            except ValueError:
                final_score_float = 0.7  # Default fallback
                        
            cursor.execute("""
                UPDATE attendance_sessions
                SET exit_time = ?, 
                    duration_minutes = ?,
                    status = 'left',
                    attendance_status = 'left_early',
                    reason_for_scoring = ?,
                    attendance_score = ?
                WHERE student_id = ? AND session_date = ? AND session_number = ?
            """, (
                current_time.strftime('%H:%M:%S'), 
                duration_minutes,
                f"EARLY_EXIT: {penalty_reason} (final_score:{final_score})",
                final_score,
                student_id,
                current_date,
                current_session_num
            ))
            
            conn.commit()

            # Display penalty analysis
            print(f"\n{'=' * 70}")
            print(f"⚠️  EARLY DEPARTURE PENALTY ANALYSIS")
            print(f"{'=' * 70}")
            print(f"Student: {name}")
            print(f"Session: {session_info['session_number']}")
            print(f"Duration: {duration_minutes}/45 minutes")
            print(f"Early Departure: {early_departure_minutes} minutes")
            print(f"Final Score: {final_score}")
            print(f"Penalty Reason: {penalty_reason}")
            print(f"{'=' * 70}\n")

            logging.info(f"⚠️ Early departure penalty for {name}: {final_score} - {penalty_reason}")
            
            return {
                'student_id': student_id,
                'duration_minutes': duration_minutes,
                'early_departure_minutes': early_departure_minutes,
                'final_score': float(final_score),
                'penalty_reason': penalty_reason,
                'auto_filled_sessions': True
            }

    except Exception as e:
        logging.error(f"❌ Enhanced exit recording error: {e}")
        return None

In [32]:
record_exit("Emily Johnson", llm_for_reasoning, 4)

17:36:14
Last session num was: 0
current session num was: 4
Checking session 1, filled_sessions: []
AI Response: on_time,1.0,assumed_present_between_recorded_sessions
Checking session 2, filled_sessions: [{'session': 1, 'status': 'on_time', 'score': 1.0, 'reason': 'assumed_present_between_recorded_sessions'}]


ERROR:root:Auto fill score failed: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kagq1dqdevabfd7mj8jfgmyx` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99865, Requested 295. Please try again in 2m18.24s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


AI Response: on_time,1.0,assumed_present_between_recorded_sessions
Checking session 3, filled_sessions: [{'session': 1, 'status': 'on_time', 'score': 1.0, 'reason': 'assumed_present_between_recorded_sessions'}, {'session': 2, 'status': 'on_time', 'score': 1.0, 'reason': 'assumed_present_between_recorded_sessions'}]


ERROR:root:❌ Enhanced exit recording error: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kagq1dqdevabfd7mj8jfgmyx` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99865, Requested 381. Please try again in 3m32.543999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


### For handling errors

In [63]:
@wrap_tool_call
def handle_tool_errors(request, handler):
    """Handle tool execution errors with custom messages."""
    try:
        return handler(request)
    except Exception as e:
        return ToolMessage(
            content=f"Tool error: Please check your input and try again. ({str(e)})",
            tool_call_id=request.tool_call["id"]
        )


### Tools

In [158]:
from langchain.tools import tool

def get_expected_students():
    """
    Get the complete list of all enrolled students with their IDs and names. 
    Returns a list of tuples like [(1, "John Smith"), (2, "Jane Doe"), ...]
    """
    try:
        with get_connection() as conn:
            cursor = conn.cursor()
            cursor.execute("""
            SELECT id, name
            FROM students
            """)
            return cursor.fetchall()
    except Error as e:
        print(e)
        return None

def get_present_students():
    """
    Get only the student IDs of students who are marked present today. 
    Returns a list of student IDs like [1, 2, 3] or "No present students".
    """
    try:
        with get_connection() as conn:
            cursor = conn.cursor()
            current_date = datetime.now().strftime('%Y-%m-%d')
            cursor.execute("""
            SELECT DISTINCT student_id
            FROM attendance_sessions
            WHERE session_date = ?
            """,(current_date,))
            result = cursor.fetchall()
            # Extract just the IDs from tuples
            return [row[0] for row in result] if result else "No present students"
    except Error as e:
        print(e)
        return None
def get_missing_students():
    """
    Get the list of students who are missing today.
    Returns a list of tuples with (id, name) for missing students.
    """
    students = get_expected_students()
    presented = get_present_students()
    
    if presented == "No present students":
        return [student[0] for student in students] # All students are missing
    

    present_ids = set(presented)
    
    return [student[0] for student in students if student[0] not in present_ids]

def notify_missing_students(missing_students):
    """
    Send notifications to missing students. 
    Input should be a list of student IDs that are absent today.
    Call this after identifying which students are missing by comparing 
    expected students with present students.
    Example: [1, 5, 7, 9] or [1, 3, 8, 12]
    """
    print(f"Notifying missing students: {missing_students}")



### Multi-tool

In [164]:
@tool
def attendance_manager(action: str):
    """
    Handle all attendance-related actions in one reliable tool.
    
    Supported actions:
    - 'get_missing': Returns list of missing student IDs
    - 'notify_missing': Finds and notifies missing students (limited to 5)
    - 'get_expected': Returns all expected students
    - 'get_present': Returns present student IDs
    """
    try:
        if action == "get_expected":
            return get_expected_students()
                
        elif action == "get_present":
            return get_present_students()
                
        elif action == "get_missing":
            return get_missing_students()
            
        elif action == "notify_missing":
            # Get missing students and notify first 5
            missing_ids = get_missing_students()
            missing_to_notify = missing_ids[:5]
            notify_missing_students(missing_to_notify)
            return f"Successfully notified {len(missing_to_notify)} missing students: {missing_to_notify}"
            
        else:
            return f"Unknown action: {action}. Use 'get_missing', 'notify_missing', 'get_expected', or 'get_present'"
            
    except Exception as e:
        return f"Error: {str(e)}"


## Things to add

When to Use Multiple Meta Tools:
### Tool 1: Attendance Manager (your existing one)

### def attendance_manager(action: str):
    """
    Handle all CORE attendance operations.
    Actions: 'get_missing', 'notify_missing', 'get_expected', 'get_present'
    """
### Tool 2: Analytics & Reports

### def analytics_manager(action: str):
    """
    Handle attendance ANALYTICS and reporting.
    Actions: 'weekly_report', 'trends', 'at_risk_students', 'predict_attendance'
    """
    # Advanced analytics, predictions, trends
### Tool 3: Student Services
  
### def student_services_manager(action: str):
    """
    Handle student INFORMATION and services.
    Actions: 'find_student', 'contact_info', 'attendance_history', 'parent_contact'
    """
    # Student lookup, contact, history
### Tool 4: System Admin

### def system_admin_manager(action: str):
    """
    Handle SYSTEM administration tasks.
    Actions: 'backup_data', 'export_reports', 'system_status', 'cleanup'
    """
    # System maintenance, backups, exports

### 1. Basic Checks
- Check current attendance status
- List present students  
- List missing students
- Get individual student status

### 2. Notifications
- Notify missing students
- Notify specific students
- Send reminders to late students
- Contact parents of absent students

### 3. Data Management  
- Get all student records
- Update attendance manually
- Add absence excuses/notes
- Mark students as present/absent
### 4. Reports
- Generate daily summary report
- Create weekly/monthly reports
- Export attendance data
- Print class rosters

### 5. Analytics
- Calculate attendance rates
- Identify attendance trends  
- Flag at-risk students
- Show class comparisons
### 6. System Operations
- Backup attendance data
- System status check
- Data cleanup/maintenance
- User management

### 7. Integration
- Sync with gradebook
- Export to school system
- Generate compliance reports
### 8. Intelligence
- Predict future attendance
- Suggest interventions
- Auto-flag patterns
- Generate insights

In [ ]:
agent = create_agent(
    model=llm,
    tools=[
        attendance_manager,      # Core attendance ops
        analytics_manager,       # Reports & analytics  
        attendance_recording_manager, # Record_entry and record_exists
        student_services_manager, # Student info & services
        system_admin_manager     # System maintenance
        
    ],
    middleware=[handle_tool_errors],
)

In [162]:

llm = ChatGroq(
    model="qwen/qwen3-32b",
    api_key=os.getenv("GROQ_API_KEY")
)

agent = create_agent(
    model=llm,
    tools=[attendance_manager],
    middleware=[handle_tool_errors],
)

# # Simple test
# response = agent.invoke({
#     "messages": [
#         {"role": "user", "content": """Get the list of presented and missing students for me"""}
#     ]
# })
# print(response["messages"][-1].content)